In [4]:
import pandas as pd

df = pd.read_csv('../data/menstrual_cycle_dataset_with_factors.csv')

print("Filas y columnas:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

df.head()

Filas y columnas: (895, 12)

Columnas:
['User ID', 'Age', 'BMI', 'Stress Level', 'Exercise Frequency', 'Sleep Hours', 'Diet', 'Cycle Start Date', 'Cycle Length', 'Period Length', 'Next Cycle Start Date', 'Symptoms']


,User ID,Age,BMI,Stress Level,Exercise Frequency,Sleep Hours,Diet,Cycle Start Date,Cycle Length,Period Length,Next Cycle Start Date,Symptoms
0,1,18,29.28,2,Moderate,5.4,Low Carb,2024-11-13 20:52:34.915012,26,7,2024-12-09 20:52:34.915012,Headache
1,1,18,29.28,2,Moderate,5.4,Low Carb,2024-12-09 20:52:34.915012,32,5,2025-01-10 20:52:34.915012,Fatigue
2,1,18,29.28,2,Moderate,5.4,Low Carb,2025-01-10 20:52:34.915012,41,7,2025-02-20 20:52:34.915012,Fatigue
3,1,18,29.28,2,Moderate,5.4,Low Carb,2025-02-20 20:52:34.915012,27,3,2025-03-19 20:52:34.915012,Fatigue
4,1,18,29.28,2,Moderate,5.4,Low Carb,2025-03-19 20:52:34.915012,42,5,2025-04-30 20:52:34.915012,Cramps


In [5]:
# Tipos de datos y valores nulos
print("Tipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nValores únicos en 'Symptoms':")
print(df['Symptoms'].unique())

print("\nValores únicos en 'Diet':")
print(df['Diet'].unique())

print("\nUsuarias únicas (User ID):", df['User ID'].nunique())

Tipos de datos:
User ID                    int64
Age                        int64
BMI                      float64
Stress Level               int64
Exercise Frequency           str
Sleep Hours              float64
Diet                         str
Cycle Start Date             str
Cycle Length               int64
Period Length              int64
Next Cycle Start Date        str
Symptoms                     str
dtype: object

Valores nulos por columna:
User ID                  0
Age                      0
BMI                      0
Stress Level             0
Exercise Frequency       0
Sleep Hours              0
Diet                     0
Cycle Start Date         0
Cycle Length             0
Period Length            0
Next Cycle Start Date    0
Symptoms                 0
dtype: int64

Valores únicos en 'Symptoms':
<StringArray>
['Headache', 'Fatigue', 'Cramps', 'Mood Swings', 'Bloating']
Length: 5, dtype: str

Valores únicos en 'Diet':
<StringArray>
['Low Carb', 'Vegetarian', 'Balanced', '

In [6]:
print(df['Symptoms'].unique())
print("\nNúmero de categorías distintas de síntomas:", df['Symptoms'].nunique())

<StringArray>
['Headache', 'Fatigue', 'Cramps', 'Mood Swings', 'Bloating']
Length: 5, dtype: str

Número de categorías distintas de síntomas: 5


In [7]:
import sqlite3

# Crear conexión (esto crea el archivo femcycle.db si no existe)
conn = sqlite3.connect('../femcycle.db')

# Cargar el dataframe a una tabla SQL llamada 'ciclos'
df.to_sql('ciclos', conn, if_exists='replace', index=False)

print("Base de datos creada y tabla 'ciclos' cargada correctamente")
conn.close()

Base de datos creada y tabla 'ciclos' cargada correctamente


In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../femcycle.db')

query = """
SELECT 
    "User ID",
    "Cycle Start Date",
    "Cycle Length",
    "Period Length"
FROM ciclos
WHERE "Cycle Length" < 21 OR "Cycle Length" > 35;
"""

anomalias = pd.read_sql(query, conn)
print("Número de ciclos anómalos encontrados:", len(anomalias))
anomalias

Número de ciclos anómalos encontrados: 503


,User ID,Cycle Start Date,Cycle Length,Period Length
0,1,2025-01-10 20:52:34.915012,41,7
1,1,2025-03-19 20:52:34.915012,42,5
2,1,2025-04-30 20:52:34.915012,41,5
3,1,2025-07-11 20:52:34.915012,48,7
4,1,2025-09-26 20:52:34.915012,47,4
...,...,...,...,...
498,99,2024-10-30 20:52:34.918906,50,7
499,100,2023-09-28 20:52:34.918945,46,3
500,100,2023-11-13 20:52:34.918945,49,5
501,100,2024-01-01 20:52:34.918945,42,4


## Finding: Cycle Length Outside the "Standard" Clinical Range

The traditional clinical benchmark considers a "normal" cycle to be between 
21-35 days, with 28 days as the reference average. However, the literature 
acknowledges significant individual variability, and some people have cycles 
that are regular but longer (functional oligomenorrhea, not necessarily 
pathological).

In this dataset, the mean cycle length is 37.4 days (median 37), with 56% of 
cycles falling above 35 days. This suggests the simulated population 
represents a profile of naturally longer cycles, not necessarily "abnormal" 
in the clinical sense.

For this reason, we adjust the outlier detection criteria: instead of using 
a fixed clinical range (21-35), we use a statistical approach based on 
percentiles (P5-P95) specific to this population, keeping the clinical 
benchmark as context rather than as an exclusion filter.

In [9]:
print(df['Cycle Length'].describe())
print("\nDistribución por rangos:")
print(pd.cut(df['Cycle Length'], bins=[0, 21, 28, 35, 100]).value_counts().sort_index())

count    895.000000
mean      37.373184
std        7.465008
min       25.000000
25%       31.000000
50%       37.000000
75%       43.000000
max       50.000000
Name: Cycle Length, dtype: float64

Distribución por rangos:
Cycle Length
(0, 21]        0
(21, 28]     136
(28, 35]     256
(35, 100]    503
Name: count, dtype: int64


In [10]:
adjusted_query = """
WITH percentiles AS (
    SELECT 
        (SELECT "Cycle Length" FROM ciclos ORDER BY "Cycle Length" LIMIT 1 OFFSET CAST(0.05 * (SELECT COUNT(*) FROM ciclos) AS INT)) AS p5,
        (SELECT "Cycle Length" FROM ciclos ORDER BY "Cycle Length" LIMIT 1 OFFSET CAST(0.95 * (SELECT COUNT(*) FROM ciclos) AS INT)) AS p95
)
SELECT 
    c."User ID",
    c."Cycle Start Date",
    c."Cycle Length",
    c."Period Length"
FROM ciclos c, percentiles p
WHERE c."Cycle Length" < p.p5 OR c."Cycle Length" > p.p95;
"""

adjusted_outliers = pd.read_sql(adjusted_query, conn)
print("Number of real outliers (P5-P95):", len(adjusted_outliers))
adjusted_outliers

Number of real outliers (P5-P95): 66


,User ID,Cycle Start Date,Cycle Length,Period Length
0,3,2024-09-29 20:52:34.915155,50,4
1,3,2024-12-28 20:52:34.915155,25,5
2,6,2023-10-19 20:52:34.915275,50,5
3,6,2024-01-23 20:52:34.915275,25,5
4,8,2023-09-23 20:52:34.915353,25,7
...,...,...,...,...
61,92,2026-01-11 20:52:34.918686,50,3
62,96,2024-04-28 20:52:34.918822,50,5
63,98,2025-05-10 20:52:34.918876,50,3
64,99,2024-05-03 20:52:34.918906,25,5


In [11]:
conn = sqlite3.connect('../femcycle.db')

In [16]:
phase_ranges_query = """
SELECT
    "User ID",
    "Cycle Start Date",
    "Cycle Length",
    "Period Length",
    "Cycle Start Date" AS menstrual_start,
    date("Cycle Start Date", '+' || ("Period Length" - 1) || ' days') AS menstrual_end,
    date("Cycle Start Date", '+' || "Period Length" || ' days') AS follicular_start,
    date("Cycle Start Date", '+' || ("Cycle Length" - 15) || ' days') AS follicular_end,
    date("Cycle Start Date", '+' || ("Cycle Length" - 14) || ' days') AS ovulation_day,
    date("Cycle Start Date", '+' || ("Cycle Length" - 13) || ' days') AS luteal_start,
    date("Cycle Start Date", '+' || ("Cycle Length" - 1) || ' days') AS luteal_end
FROM ciclos;
"""

phase_ranges = pd.read_sql(phase_ranges_query, conn)
phase_ranges

,User ID,Cycle Start Date,Cycle Length,Period Length,menstrual_start,menstrual_end,follicular_start,follicular_end,ovulation_day,luteal_start,luteal_end
0,1,2024-11-13 20:52:34.915012,26,7,2024-11-13 20:52:34.915012,2024-11-19,2024-11-20,2024-11-24,2024-11-25,2024-11-26,2024-12-08
1,1,2024-12-09 20:52:34.915012,32,5,2024-12-09 20:52:34.915012,2024-12-13,2024-12-14,2024-12-26,2024-12-27,2024-12-28,2025-01-09
2,1,2025-01-10 20:52:34.915012,41,7,2025-01-10 20:52:34.915012,2025-01-16,2025-01-17,2025-02-05,2025-02-06,2025-02-07,2025-02-19
3,1,2025-02-20 20:52:34.915012,27,3,2025-02-20 20:52:34.915012,2025-02-22,2025-02-23,2025-03-04,2025-03-05,2025-03-06,2025-03-18
4,1,2025-03-19 20:52:34.915012,42,5,2025-03-19 20:52:34.915012,2025-03-23,2025-03-24,2025-04-15,2025-04-16,2025-04-17,2025-04-29
...,...,...,...,...,...,...,...,...,...,...,...
890,100,2023-08-24 20:52:34.918945,35,5,2023-08-24 20:52:34.918945,2023-08-28,2023-08-29,2023-09-13,2023-09-14,2023-09-15,2023-09-27
891,100,2023-09-28 20:52:34.918945,46,3,2023-09-28 20:52:34.918945,2023-09-30,2023-10-01,2023-10-29,2023-10-30,2023-10-31,2023-11-12
892,100,2023-11-13 20:52:34.918945,49,5,2023-11-13 20:52:34.918945,2023-11-17,2023-11-18,2023-12-17,2023-12-18,2023-12-19,2023-12-31
893,100,2024-01-01 20:52:34.918945,42,4,2024-01-01 20:52:34.918945,2024-01-04,2024-01-05,2024-01-28,2024-01-29,2024-01-30,2024-02-11


In [17]:
print(phase_ranges.shape)

(895, 11)
